In [0]:
SELECT COUNT(*) AS session_count
, SUM(ts_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM (
  SELECT tvid, ts_start, ts_end, cid
  , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
  , LEAD(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_epid
  , LAG(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_epid
  , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
  FROM prod.cooker.vizio_content_firehose
  WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
    )
WHERE next_start = ts_end
  AND prev_end = ts_start
  AND LOWER(cid) = 'unknown'
  AND LOWER(next_epid) != 'unknown'
  AND LOWER(prev_epid) != 'unknown'
  AND next_epid = prev_epid
  AND ts_duration <= 180

In [0]:
SELECT COUNT(*) AS session_count
, SUM(TIMESTAMPDIFF(SECOND, ts_start, ts_end))/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM prod.cooker.vizio_content_firehose
WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
  AND LOWER(cid) = 'unknown'

In [0]:
SELECT COUNT(*) AS session_count
, SUM(ts_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM (
  SELECT tvid, ts_start, ts_end, cid
  , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
  , LEAD(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_epid
  , LAG(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_epid
  , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
  FROM prod.cooker.vizio_content_firehose
  WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE TIMESTAMPDIFF(SECOND, ts_end, next_start) <= 2
  AND TIMESTAMPDIFF(SECOND, prev_end, ts_start) <= 2
  AND LOWER(cid) = 'unknown'
  AND LOWER(next_epid) != 'unknown'
  AND LOWER(prev_epid) != 'unknown'
  AND next_epid = prev_epid
  AND ts_duration <= 180

In [0]:
SELECT tvid, ts_start, ts_end, cid
FROM (
  SELECT tvid, ts_start, ts_end, cid
  , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
  , LEAD(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_epid
  , LAG(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_epid
  , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
  FROM prod.cooker.vizio_content_firehose
  WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE TIMESTAMPDIFF(SECOND, ts_end, next_start) <= 2
  AND TIMESTAMPDIFF(SECOND, prev_end, ts_start) <= 2
  AND LOWER(cid) = 'unknown'
  AND LOWER(next_epid) != 'unknown'
  AND LOWER(prev_epid) != 'unknown'
  AND next_epid = prev_epid
  AND ts_duration <= 180
ORDER BY 1, 2, 3
LIMIT 100

In [0]:
SELECT tvid, tv_group_num, MIN(ts_start) AS ts_start, MAX(ts_end) AS ts_end
FROM (
  SELECT *
  , SUM(tvog_calc) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
  FROM (
    SELECT tvid, ts_start, ts_end
    , MAX(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
    , CASE WHEN TIMESTAMPADD(SECOND, -2, ts_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
    FROM (
      SELECT tvid, ts_start, ts_end
      FROM prod.cooker.vizio_attrcomm_firehose
      GROUP BY ALL
      ORDER BY 1, 2, 3
    )
    WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
      AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
      AND tvid = 4001275
  )
)
GROUP BY 1, 2
ORDER BY 1, 2, 3
LIMIT 100

In [0]:
SELECT 400.379444/2243968.445556

In [0]:
WITH null_gaps_to_fill AS (
  SELECT tvid, ts_start, ts_end, cid, ts_duration
  FROM (
    SELECT tvid, ts_start, ts_end, cid
    , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
    , LEAD(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_epid
    , LAG(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_epid
    , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
    FROM prod.cooker.vizio_content_firehose
    WHERE ts_start >= CURRENT_TIMESTAMP - INTERVAL 3 HOURS
      AND ts_start < CURRENT_TIMESTAMP - INTERVAL 2 HOURS
  )
  WHERE TIMESTAMPDIFF(SECOND, ts_end, next_start) <= 2
    AND TIMESTAMPDIFF(SECOND, prev_end, ts_start) <= 2
    AND LOWER(cid) = 'unknown'
    AND LOWER(next_epid) != 'unknown'
    AND LOWER(prev_epid) != 'unknown'
    AND next_epid = prev_epid
    AND ts_duration <= 180
)
, tvids_from_it AS (
  SELECT tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT tvid, tv_group_num, MIN(ts_start) AS ts_start, MAX(ts_end) AS ts_end
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT tvid, ts_start, ts_end
      , MAX(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, ts_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.tvid, vc.ts_start, vc.ts_end
        FROM prod.cooker.vizio_attrcomm_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.tvid = vc.tvid
        WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
          AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      )
    )
  )
  GROUP BY 1, 2
)
SELECT content.*, comms.ts_start AS comm_start, comms.ts_end AS comm_end
FROM null_gaps_to_fill AS content
JOIN comms
  ON content.tvid = comms.tvid
 AND comms.ts_start <= content.ts_end
 AND comms.ts_end >= content.ts_start
GROUP BY ALL
ORDER BY 1, 2, 3, 6, 7
LIMIT 100

In [0]:
WITH null_gaps_to_fill AS (
  SELECT tvid, ts_start, ts_end, cid, ts_duration
  FROM (
    SELECT tvid, ts_start, ts_end, cid
    , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
    , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
    , LEAD(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_epid
    , LAG(epid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_epid
    , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
    FROM prod.cooker.vizio_content_firehose
    WHERE ts_start >= CURRENT_TIMESTAMP - INTERVAL 3 HOURS
      AND ts_start < CURRENT_TIMESTAMP - INTERVAL 2 HOURS
  )
  WHERE TIMESTAMPDIFF(SECOND, ts_end, next_start) <= 2
    AND TIMESTAMPDIFF(SECOND, prev_end, ts_start) <= 2
    AND LOWER(cid) = 'unknown'
    AND LOWER(next_epid) != 'unknown'
    AND LOWER(prev_epid) != 'unknown'
    AND next_epid = prev_epid
    AND ts_duration <= 180
)
, tvids_from_it AS (
  SELECT tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT tvid, tv_group_num, MIN(ts_start) AS ts_start, MAX(ts_end) AS ts_end
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT tvid, ts_start, ts_end
      , MAX(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start, ts_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, ts_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.tvid, vc.ts_start, vc.ts_end
        FROM prod.cooker.vizio_attrcomm_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.tvid = vc.tvid
        WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
          AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      )
    )
  )
  GROUP BY 1, 2
)
SELECT COUNT(*) AS session_count
, SUM(ts_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM (
SELECT content.*
FROM null_gaps_to_fill AS content
JOIN comms
  ON content.tvid = comms.tvid
 AND comms.ts_start <= content.ts_end
 AND comms.ts_end >= content.ts_start
GROUP BY ALL)

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
  -- AND (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
  AND fk_show_id IS NULL
  AND (next_show_id IS NOT NULL OR (next_show_id IS NULL AND prev_show_id IS NOT NULL))
  AND session_duration <= 180

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
-- WHERE (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
WHERE (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
  AND fk_show_id IS NULL
  AND (prev_show_id IS NOT NULL OR (prev_show_id IS NULL AND next_show_id IS NOT NULL))
  AND session_duration <= 180

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE fk_show_id IS NULL
  AND session_duration <= 180
  AND (
    (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
      OR (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
      )
  AND (
    (prev_show_id IS NOT NULL OR (prev_show_id IS NULL AND next_show_id IS NOT NULL))
    OR (next_show_id IS NOT NULL OR (next_show_id IS NULL AND prev_show_id IS NOT NULL))
    )

In [0]:
SELECT *
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE fk_show_id IS NULL
  AND session_duration <= 180
  AND prev_show_id OR next_show_id IS NOT NULL
  -- AND (
  --   (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
  --     OR (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
  --     )
  -- AND (
  --   (prev_show_id IS NOT NULL OR (prev_show_id IS NULL AND next_show_id IS NOT NULL))
  --   OR (next_show_id IS NOT NULL OR (next_show_id IS NULL AND prev_show_id IS NOT NULL))
  --   )
ORDER BY 1, 2
LIMIT 1000

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
  , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE fk_show_id IS NULL
  AND session_duration <= 180
  AND (prev_show_id IS NOT NULL OR next_show_id IS NOT NULL)
  -- AND (
  --   (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
  --     OR (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
  --     )
  -- AND (
  --   (prev_show_id IS NOT NULL OR (prev_show_id IS NULL AND next_show_id IS NOT NULL))
  --   OR (next_show_id IS NOT NULL OR (next_show_id IS NULL AND prev_show_id IS NOT NULL))
  --   )

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
  , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE fk_show_id IS NULL
  AND session_duration <= 180
  AND ((prev_show_id IS NOT NULL AND prev_is_live) OR (next_show_id IS NOT NULL AND next_is_live))
  -- AND (
  --   (next_start = session_end OR (next_start IS NULL AND DATE_PART('MINUTE', session_end) >= 57))
  --     OR (prev_end = session_start OR (prev_end IS NULL AND DATE_PART('MINUTE', session_start) <= 3))
  --     )
  -- AND (
  --   (prev_show_id IS NOT NULL OR (prev_show_id IS NULL AND next_show_id IS NOT NULL))
  --   OR (next_show_id IS NOT NULL OR (next_show_id IS NULL AND prev_show_id IS NOT NULL))
  --   )

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM prod.detection.viewing_content_firehose
WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
  AND fk_show_id IS NULL

In [0]:
SELECT COUNT(*) AS session_count
, SUM(ts_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM (
  SELECT tvid, ts_start, ts_end, cid
  , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
  , LEAD(cid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_cid
  , LAG(cid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_cid
  , LEAD(is_live) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_is_live
  , LAG(is_live) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_is_live
  , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
  FROM prod.cooker.vizio_content_firehose
  WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 2 HOURS
)
WHERE cid = 'unknown'
  AND ts_duration <= 180
  AND ((prev_cid IS NOT NULL AND prev_cid != 'unknown' AND prev_is_live) OR (next_cid IS NOT NULL AND next_cid != 'unknonw' AND next_is_live))

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
  , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
  , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
  , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
  , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
  , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
  , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
  FROM prod.detection.viewing_content_firehose
  WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
    AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
)
WHERE fk_show_id IS NULL
  AND session_duration <= 180
  AND (
    (prev_show_id IS NOT NULL AND prev_is_live)
    OR (next_show_id IS NOT NULL AND next_is_live)
    )

In [0]:
/*SELECT COUNT(*) AS session_count
, SUM(ts_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT tvid) AS tv_count
FROM (
  SELECT tvid, ts_start, ts_end, cid
  , lead(ts_start) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_start
  , LAG(ts_end) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_end
  , LEAD(cid) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_cid
  , LAG(cid) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_cid
  , LEAD(is_live) OVER (PARTITION BY tvid ORDER BY ts_start) AS next_is_live
  , LAG(is_live) OVER (PARTITION BY tvid ORDER BY ts_start) AS prev_is_live
  , TIMESTAMPDIFF(SECOND, ts_start, ts_end) AS ts_duration
  FROM prod.cooker.vizio_content_firehose
  WHERE ts_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
    AND ts_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
)
WHERE cid = 'unknown'
  AND ts_duration <= 180
  AND ((prev_cid IS NOT NULL AND prev_cid != 'unknown' AND prev_is_live) OR (next_cid IS NOT NULL AND next_cid != 'unknown' AND next_is_live))

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, airdate
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND ((prev_show_id IS NOT NULL) OR (next_show_id IS NOT NULL))
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num, MIN(session_start) AS session_start, MAX(session_end) AS session_end
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      )
    )
  )
  GROUP BY 1, 2
)
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
SELECT content.*
FROM null_gaps_to_fill AS content
JOIN comms
  ON content.fk_tvid = comms.fk_tvid
 AND comms.session_start <= content.session_end
 AND comms.session_end >= content.session_start
GROUP BY ALL)

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND (
      (prev_show_id IS NOT NULL AND prev_is_live AND prev_input_source_id = fk_input_source_id)
      OR
      (next_show_id IS NOT NULL AND next_is_live AND next_input_source_id = fk_input_source_id)
    )
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num, MIN(session_start) AS session_start, MAX(session_end) AS session_end
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      )
    )
  )
  GROUP BY 1, 2
)
-- SELECT COUNT(*) AS session_count
-- , SUM(session_duration)/3600.0 AS ttl_duration
-- , COUNT(DISTINCT fk_tvid) AS tv_count
-- FROM (
  SELECT content.*, comms.session_start AS comm_start, comms.session_end AS comm_end
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  GROUP BY ALL
  ORDER BY 1, 2, 5
  LIMIT 1000
-- )

In [0]:
SELECT * FROM prod.detection.viewing_content_firehose
WHERE fk_tvid = 3339441
AND DATE_TRUNC('HOUR', session_start) >= '2025-07-01 17:00:00'
ORDER BY session_start

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id, media_time_start, media_time_end
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND (
      (prev_show_id IS NOT NULL AND prev_is_live AND prev_input_source_id = fk_input_source_id)
      OR
      (next_show_id IS NOT NULL AND next_is_live AND next_input_source_id = fk_input_source_id)
    )
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num, MIN(session_start) AS session_start, MAX(session_end) AS session_end
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      )
    )
  )
  GROUP BY 1, 2
)
-- SELECT COUNT(*) AS session_count
-- , SUM(session_duration)/3600.0 AS ttl_duration
-- , COUNT(DISTINCT fk_tvid) AS tv_count
-- FROM (
  SELECT content.*, comms.session_start AS comm_start, comms.session_end AS comm_end
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  GROUP BY ALL
  ORDER BY 1, 2, 5
  LIMIT 1000
-- )

In [0]:
SELECT COUNT(*) AS session_count, SUM(session_duration)/3600.0 AS ttl_duration
FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
    , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
          THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
          AND prev_media_time_end + session_duration <= next_media_time_start
          AND prev_is_live <=> next_is_live
          AND prev_is_live
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
          THEN prev_input_source_id <=> fk_input_source_id
          AND prev_media_time_end + session_duration <= prev_runtime
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
          THEN next_input_source_id <=> fk_input_source_id
           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
           AND next_media_time_start - session_duration >= 0
          AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
          THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                      THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                      THEN next_input_source_id <=> fk_input_source_id
                       AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                       AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
  -- GROUP BY ALL
-- ORDER BY fk_tvid, session_start
-- LIMIT 100

In [0]:
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
FROM prod.detection.viewing_content_firehose
WHERE session_start >= CURRENT_DATE - 1
  AND session_start < CURRENT_DATE
  AND fk_show_id IS NULL

In [0]:
SELECT fk_tvid, session_start, session_end, session_duration
FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
    , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
          THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
          AND prev_media_time_end + session_duration <= next_media_time_start
          AND prev_is_live <=> next_is_live
          AND prev_is_live
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
          THEN prev_input_source_id <=> fk_input_source_id
          AND prev_media_time_end + session_duration <= prev_runtime
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
          THEN next_input_source_id <=> fk_input_source_id
           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
           AND next_media_time_start - session_duration >= 0
          AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
          THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                      THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                      THEN next_input_source_id <=> fk_input_source_id
                       AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                       AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
  GROUP BY ALL
ORDER BY fk_tvid, session_start
LIMIT 100

In [0]:
SELECT fk_tvid, fk_show_id, fk_station_id
, REPLACE(airdate, '.000+00:00', '') AS airdate
, REPLACE(session_start, '.000+00:00', '') AS session_start
, REPLACE(session_end, '.000+00:00', '') AS session_end
, session_duration
, media_time_start, media_time_end, runtime
, fk_input_source_id
, is_live
FROM prod.detection.viewing_content_firehose
WHERE fk_tvid = 3332343
AND DATE_TRUNC('HOUR', session_start) = DATE_TRUNC('HOUR', '2025-07-10T13:00:00')
ORDER BY session_start

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
    , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
          THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
          AND prev_media_time_end + session_duration <= next_media_time_start
          AND prev_is_live <=> next_is_live
          AND prev_is_live
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
          THEN prev_input_source_id <=> fk_input_source_id
          AND prev_media_time_end + session_duration <= prev_runtime
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
          THEN next_input_source_id <=> fk_input_source_id
           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
           AND next_media_time_start - session_duration >= 0
           AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
          THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                      THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                      THEN next_input_source_id <=> fk_input_source_id
                       AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                       AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num, MIN(session_start) AS session_start, MAX(session_end) AS session_end, TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      ) 
    )
  )
  GROUP BY 1, 2
)
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
FROM (
  SELECT content.*--, comms.session_start AS comm_start, comms.session_end AS comm_end
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  GROUP BY ALL
  -- ORDER BY 1, 2, 5
  -- LIMIT 1000
)

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  , CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
         WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
         WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
         WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
              THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime
                         AND prev_input_source_id <=> fk_input_source_id
                         AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                         AND prev_is_live THEN 2
                        WHEN next_media_time_start - session_duration >= 0
                         AND next_input_source_id <=> fk_input_source_id
                         AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                         AND next_is_live THEN 3 END END AS num_for_calc
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_is_live
    , LAG(is_live) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_is_live
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
    , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
    , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
    , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
    , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
    , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
    , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
    , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= CURRENT_DATE - 1
      AND session_start < CURRENT_DATE
    -- WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
    --   AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND next_start >= session_end
    AND prev_end <= session_start
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
          THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
          AND prev_media_time_end + session_duration <= next_media_time_start
          AND next_is_live
          AND prev_is_live
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
          THEN prev_input_source_id <=> fk_input_source_id
          AND prev_media_time_end + session_duration <= prev_runtime
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
          THEN next_input_source_id <=> fk_input_source_id
           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
           AND next_media_time_start - session_duration >= 0
           AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
          THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                      THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                      THEN next_input_source_id <=> fk_input_source_id
                       AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                       AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num
  , MIN(session_start) AS session_start
  , MAX(session_end) AS session_end
  , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= CURRENT_DATE - 1
          AND session_start < CURRENT_DATE
        -- WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
        --   AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      ) 
    )
  )
  GROUP BY 1, 2
)
SELECT COUNT(*) AS session_count
, SUM(session_duration)/3600.0 AS ttl_duration
, COUNT(DISTINCT fk_tvid) AS tv_count
-- SELECT *, ROUND(ttl_comm_duration/session_duration*100.0, 2) AS ratio
FROM (
  SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, SUM(comms.comm_duration)*1.0 AS ttl_comm_duration
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  GROUP BY 1, 2, 3, 4
  -- LIMIT 1000
)
WHERE CASE WHEN session_duration <= 30 THEN ttl_comm_duration >= 5
           ELSE ttl_comm_duration/session_duration >= 0.5 END
-- ORDER BY 1, 2, 3, 4
LIMIT 1000

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid, session_start, session_end, session_duration
  -- , CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
  --        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
  --        WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
  --        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
  --             THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime
  --                        AND prev_input_source_id <=> fk_input_source_id
  --                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
  --                        AND prev_is_live THEN 2
  --                       WHEN next_media_time_start - session_duration >= 0
  --                        AND next_input_source_id <=> fk_input_source_id
  --                        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
  --                        AND next_is_live THEN 3 END END AS num_for_calc
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
    , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
    , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
    , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
    , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
    , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
    , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
    , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
    , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
    , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
    , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
    , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
    , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
    , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
    , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
    , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
    , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
    , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
    FROM prod.detection.viewing_content_firehose
    WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
      AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
  )
  WHERE fk_show_id IS NULL
    AND session_duration <= 180
    AND next_start >= session_end
    AND prev_end <= session_start
    AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
          THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
          AND prev_media_time_end + session_duration <= next_media_time_start
          AND next_is_live
          AND prev_is_live
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
          THEN prev_input_source_id <=> fk_input_source_id
          AND prev_media_time_end + session_duration <= prev_runtime
          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
          AND prev_is_live
        WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
          THEN next_input_source_id <=> fk_input_source_id
           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
           AND next_media_time_start - session_duration >= 0
           AND next_is_live
        WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
          THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                      THEN prev_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                        AND prev_is_live
                    WHEN next_media_time_start - session_duration >= 0
                      THEN next_input_source_id <=> fk_input_source_id
                       AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                       AND next_is_live
                    ELSE FALSE END
        ELSE FALSE END
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num
  , MIN(session_start) AS session_start
  , MAX(session_end) AS session_end
  , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      ) 
    )
  )
  GROUP BY 1, 2
)
SELECT *
, ttl_comm_duration/session_duration AS ratio
FROM (
  SELECT content.fk_tvid
  , content.session_start
  , content.session_end
  , content.session_duration
  , SUM(comms.comm_duration)*1.0 AS ttl_comm_duration
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  GROUP BY 1, 2, 3, 4
)
WHERE CASE WHEN session_duration <= 30 THEN ttl_comm_duration >= 5
           ELSE ttl_comm_duration/session_duration >= 0.5 END
ORDER BY 1, 2, 3, 4
LIMIT 1000

In [0]:
WITH null_gaps_to_fill AS (
  SELECT fk_tvid
  , session_start
  , session_end
  , session_duration
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
         WHEN num_for_calc = 3 THEN next_media_time_start - session_duration
    END AS media_time_start
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
         WHEN num_for_calc = 3 THEN next_media_time_start
    END AS media_time_end
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
         WHEN num_for_calc = 3 THEN next_station_id
    END AS fk_station_id
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
         WHEN num_for_calc = 3 THEN next_show_id
    END AS fk_show_id
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
         WHEN num_for_calc = 3 THEN next_airdate
    END AS airdate
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
         WHEN num_for_calc = 3 THEN next_runtime
    END AS runtime

  , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
         WHEN num_for_calc = 3 THEN next_tms_airdate
    END AS tms_airdate
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
         WHEN num_for_calc = 3 THEN next_tms_station_id
    END AS tms_station_id
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
         WHEN num_for_calc = 3 THEN next_tms_show_id
    END AS tms_show_id
  , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
         WHEN num_for_calc = 3 THEN next_fk_content_id
    END AS fk_content_id
  , TRUE AS is_live

  , fk_input_source_id
  FROM (
    SELECT *
    , CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
           WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
           WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
           WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                           AND next_input_source_id <=> fk_input_source_id
                           AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                           AND NVL(next_is_live, FALSE) THEN 3
                          WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                           AND prev_input_source_id <=> fk_input_source_id
                           AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                           AND NVL(prev_is_live, FALSE) THEN 2
                    END
      END AS num_for_calc
    FROM (
      SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
      , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
      , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
      , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
      , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
      , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
      , LEAD(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_station_id
      , LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_station_id
      , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
      , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
      , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
      , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
      , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
      , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
      , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
      , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
      , LEAD(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_airdate
      , LAG(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_airdate
      , LEAD(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_station_id
      , LAG(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_station_id
      , LEAD(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_show_id
      , LAG(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_show_id
      , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_fk_content_id
      , LAG(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_fk_content_id
      , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
      , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
      , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
      , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
      FROM prod.detection.viewing_content_firehose
      WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
        AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    )
    WHERE fk_show_id IS NULL
      AND session_duration <= 180
      AND next_start >= session_end
      AND prev_end <= session_start
      AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
            THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
            AND prev_media_time_end + session_duration <= next_media_time_start
            AND next_is_live
            AND prev_is_live
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
          WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
            THEN prev_input_source_id <=> fk_input_source_id
            AND prev_media_time_end + session_duration <= prev_runtime
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND prev_is_live
          WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
            THEN next_input_source_id <=> fk_input_source_id
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            AND next_media_time_start - session_duration >= 0
            AND next_is_live
          WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
            THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                        THEN prev_input_source_id <=> fk_input_source_id
                          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                          AND prev_is_live
                      WHEN next_media_time_start - session_duration >= 0
                        THEN next_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                        AND next_is_live
                      ELSE FALSE END
          ELSE FALSE END
    GROUP BY ALL
  )
  GROUP BY ALL
)
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num
  , MIN(session_start) AS session_start
  , MAX(session_end) AS session_end
  , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      ) 
    )
  )
  GROUP BY 1, 2
)
SELECT *
FROM (
SELECT fk_tvid, session_start, session_end, session_duration
  , media_time_start, media_time_end
  , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
  , SUM(comm_duration)*1.0 AS total_comm_duration
FROM (
  SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration
  , content.media_time_start, content.media_time_end, content.fk_show_id, content.airdate
  , content.runtime, content.fk_input_source_id
  , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
  , GREATEST(comms.session_start, content.session_start) AS comm_start
  , LEAST(comms.session_end, content.session_end) AS comm_end
  , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
  FROM null_gaps_to_fill AS content
  JOIN comms
    ON content.fk_tvid = comms.fk_tvid
  AND comms.session_start <= content.session_end
  AND comms.session_end >= content.session_start
  WHERE content.fk_show_id IS NOT NULL
    AND content.airdate IS NOT NULL
    AND content.media_time_start IS NOT NULL
    AND content.fk_station_id IS NOT NULL
)
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14)
WHERE CASE WHEN session_duration <= 15 THEN total_comm_duration >= 5
           ELSE total_comm_duration/session_duration >= 0.5 END
ORDER BY 1, 2, 3, 4
LIMIT 1000

In [0]:
WITH filled_null_gaps AS (
  WITH null_gaps_to_fill AS (
    SELECT fk_tvid
    , session_start
    , session_end
    , session_duration
    , fk_input_source_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
           WHEN num_for_calc = 3 THEN next_media_time_start - session_duration END AS media_time_start
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
           WHEN num_for_calc = 3 THEN next_media_time_start END AS media_time_end
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
           WHEN num_for_calc = 3 THEN next_station_id END AS fk_station_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
           WHEN num_for_calc = 3 THEN next_show_id END AS fk_show_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
           WHEN num_for_calc = 3 THEN next_airdate END AS airdate
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
          WHEN num_for_calc = 3 THEN next_runtime END AS runtime
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
           WHEN num_for_calc = 3 THEN next_tms_airdate END AS tms_airdate
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
           WHEN num_for_calc = 3 THEN next_tms_station_id END AS tms_station_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
           WHEN num_for_calc = 3 THEN next_tms_show_id END AS tms_show_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
           WHEN num_for_calc = 3 THEN next_fk_content_id END AS fk_content_id
    , TRUE AS is_live

    , fk_input_source_id
    FROM (
      SELECT *
      , CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
            WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                  THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                            AND next_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                            AND NVL(next_is_live, FALSE) THEN 3
                            WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                            AND prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND NVL(prev_is_live, FALSE) THEN 2
                      END
        END AS num_for_calc
      FROM (
        SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
        , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
        , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
        , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
        , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
        , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
        , LEAD(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_station_id
        , LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_station_id
        , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
        , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
        , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
        , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
        , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
        , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
        , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
        , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
        , LEAD(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_airdate
        , LAG(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_airdate
        , LEAD(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_station_id
        , LAG(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_station_id
        , LEAD(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_show_id
        , LAG(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_show_id
        , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_fk_content_id
        , LAG(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_fk_content_id
        , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
        , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
        , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
        , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
        FROM prod.detection.viewing_content_firehose
        WHERE session_start >= '2025-08-04 00:00:00'
          AND session_start < '2025-08-07 00:00:00'
          AND fk_tvid = 170122240
      )
      WHERE fk_show_id IS NULL
        AND session_duration <= 180
        AND next_start >= session_end
        AND prev_end <= session_start
        AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
              THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
              AND prev_media_time_end + session_duration <= next_media_time_start
              AND next_is_live
              AND prev_is_live
              AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
              AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
              THEN prev_input_source_id <=> fk_input_source_id
              AND prev_media_time_end + session_duration <= prev_runtime
              AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
              AND prev_is_live
            WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
              THEN next_input_source_id <=> fk_input_source_id
              AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
              AND next_media_time_start - session_duration >= 0
              AND next_is_live
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
              THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                          THEN prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND prev_is_live
                        WHEN next_media_time_start - session_duration >= 0
                          THEN next_input_source_id <=> fk_input_source_id
                          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                          AND next_is_live
                        ELSE FALSE END
            ELSE FALSE END
      GROUP BY ALL
    )
    GROUP BY ALL
  )
  -- , tvids_from_it AS (
  --   SELECT fk_tvid
  --   FROM null_gaps_to_fill
  --   GROUP BY 1
  -- )
  , comms AS (
    SELECT fk_tvid, tv_group_num
    , MIN(session_start) AS session_start
    , MAX(session_end) AS session_end
    , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
    FROM (
      SELECT *
      , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
      FROM (
        SELECT fk_tvid, session_start, session_end
        , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
        , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
        FROM (
          SELECT comms.fk_tvid, comms.session_start, comms.session_end
          FROM prod.detection.viewing_commercials_firehose AS comms
          JOIN null_gaps_to_fill AS tv
            ON tv.fk_tvid = comms.fk_tvid
           AND comms.session_start <= tv.session_end
           AND comms.session_end >= tv.session_start
          WHERE comms.session_start >= '2025-08-04 00:00:00'
            AND comms.session_start < '2025-08-07 18:00:00'
            AND comms.fk_tvid = 170122240
          GROUP BY ALL
          ORDER BY 1, 2, 3
        ) 
      )
    )
    GROUP BY 1, 2
  )
  SELECT *
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_input_source_id
      , media_time_start, media_time_end, is_live
      , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
      , SUM(comm_duration)*1.0 AS total_comm_duration
    FROM (
      SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, content.fk_input_source_id
      , content.media_time_start, content.media_time_end, content.is_live, content.fk_show_id, content.airdate
      , content.runtime, content.fk_input_source_id
      , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
      , GREATEST(comms.session_start, content.session_start) AS comm_start
      , LEAST(comms.session_end, content.session_end) AS comm_end
      , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
      FROM null_gaps_to_fill AS content
      JOIN comms
        ON content.fk_tvid = comms.fk_tvid
      AND comms.session_start <= content.session_end
      AND comms.session_end >= content.session_start
      WHERE content.fk_show_id IS NOT NULL
        AND content.airdate IS NOT NULL
        AND content.media_time_start IS NOT NULL
        AND content.fk_station_id IS NOT NULL
    )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16
  )
  WHERE CASE WHEN session_duration <= 15 THEN total_comm_duration >= 5
            ELSE total_comm_duration/session_duration >= 0.5 END
)
SELECT c.fk_tvid
, c.session_start
, c.session_end
, COALESCE(n.session_duration, c.session_duration) AS session_duration
, COALESCE(n.fk_show_id, c.fk_show_id) AS fk_show_id
, COALESCE(n.fk_station_id, c.fk_station_id) AS fk_station_id
, COALESCE(n.airdate, c.airdate) AS airdate
, COALESCE(n.tms_airdate, c.tms_airdate) AS tms_airdate
, COALESCE(n.tms_show_id, c.tms_show_id) AS tms_show_id
, COALESCE(n.tms_station_id, c.tms_station_id) AS tms_station_id
, COALESCE(n.media_time_start, c.media_time_start) AS media_time_start
, COALESCE(n.runtime, c.runtime) AS runtime
, COALESCE(n.fk_content_id, c.fk_content_id) AS fk_content_id
, c.fk_input_source_id
, c.fk_dma_id
, c.fk_location_id
, COALESCE(n.is_live, c.is_live) AS is_live
, c.fk_zoo_id
, c.tms_tuner_channel_id
, c.tms_tuner_program_id
, c.tuner_channel_id
, c.tuner_program_id
, c.vizio_epg_program
, c.vizio_epg_station
, c.file_ingested
, CASE WHEN n.fk_tvid IS NOT NULL THEN 1 END AS if_filled
FROM prod.detection.viewing_content_firehose c
LEFT JOIN filled_null_gaps n
  ON n.fk_tvid = c.fk_tvid
 AND n.session_start = c.session_start
 AND n.session_end = c.session_end
 AND n.fk_input_source_id = c.fk_input_source_id
WHERE c.session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
  AND c.session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
ORDER BY 1, 2, 3
LIMIT 1000

In [0]:
WITH filled_null_gaps AS (
  WITH null_gaps_to_fill AS (
    SELECT fk_tvid
    , session_start
    , session_end
    , session_duration
    , fk_input_source_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end
           WHEN num_for_calc = 3 THEN next_media_time_start - session_duration END AS media_time_start
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_media_time_end + session_duration
           WHEN num_for_calc = 3 THEN next_media_time_start END AS media_time_end
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_station_id
           WHEN num_for_calc = 3 THEN next_station_id END AS fk_station_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_show_id
           WHEN num_for_calc = 3 THEN next_show_id END AS fk_show_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_airdate
           WHEN num_for_calc = 3 THEN next_airdate END AS airdate
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_runtime
          WHEN num_for_calc = 3 THEN next_runtime END AS runtime
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_airdate
           WHEN num_for_calc = 3 THEN next_tms_airdate END AS tms_airdate
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_station_id
           WHEN num_for_calc = 3 THEN next_tms_station_id END AS tms_station_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_tms_show_id
           WHEN num_for_calc = 3 THEN next_tms_show_id END AS tms_show_id
    , CASE WHEN num_for_calc IN (1, 2) THEN prev_fk_content_id
           WHEN num_for_calc = 3 THEN next_fk_content_id END AS fk_content_id
    , TRUE AS is_live

    , fk_input_source_id
    FROM (
      SELECT *
      , CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id THEN 1
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL THEN 2
            WHEN next_show_id IS NOT NULL AND next_show_id IS NULL THEN 3
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
                  THEN CASE WHEN NVL(next_media_time_start, 0) - session_duration >= 0
                            AND next_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                            AND NVL(next_is_live, FALSE) THEN 3
                            WHEN NVL(prev_media_time_end, 0) + session_duration <= NVL(prev_runtime, 0)
                            AND prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND NVL(prev_is_live, FALSE) THEN 2
                      END
        END AS num_for_calc
      FROM (
        SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
        , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
        , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
        , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
        , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
        , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
        , LEAD(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_station_id
        , LAG(fk_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_station_id
        , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
        , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
        , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
        , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
        , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
        , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
        , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
        , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
        , LEAD(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_airdate
        , LAG(tms_airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_airdate
        , LEAD(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_station_id
        , LAG(tms_station_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_station_id
        , LEAD(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_tms_show_id
        , LAG(tms_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_tms_show_id
        , LEAD(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_fk_content_id
        , LAG(fk_content_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_fk_content_id
        , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
        , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
        , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
        , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
        FROM prod.detection.viewing_content_firehose
        WHERE session_start >= '2025-08-04 00:00:00'
          AND session_start < '2025-08-07 00:00:00'
          AND fk_tvid = 170122240
      )
      WHERE fk_show_id IS NULL
        AND session_duration <= 180
        AND next_start >= session_end
        AND prev_end <= session_start
        AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
              THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
              AND prev_media_time_end + session_duration <= next_media_time_start
              AND next_is_live
              AND prev_is_live
              AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
              AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
              THEN prev_input_source_id <=> fk_input_source_id
              AND prev_media_time_end + session_duration <= prev_runtime
              AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
              AND prev_is_live
            WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
              THEN next_input_source_id <=> fk_input_source_id
              AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
              AND next_media_time_start - session_duration >= 0
              AND next_is_live
            WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
              THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                          THEN prev_input_source_id <=> fk_input_source_id
                            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                            AND prev_is_live
                        WHEN next_media_time_start - session_duration >= 0
                          THEN next_input_source_id <=> fk_input_source_id
                          AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                          AND next_is_live
                        ELSE FALSE END
            ELSE FALSE END
      GROUP BY ALL
    )
    GROUP BY ALL
  )
  -- , tvids_from_it AS (
  --   SELECT fk_tvid
  --   FROM null_gaps_to_fill
  --   GROUP BY 1
  -- )
  , comms AS (
    SELECT fk_tvid, tv_group_num
    , MIN(session_start) AS session_start
    , MAX(session_end) AS session_end
    , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
    FROM (
      SELECT *
      , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
      FROM (
        SELECT fk_tvid, session_start, session_end
        , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
        , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
        FROM (
          SELECT comms.fk_tvid, comms.session_start, comms.session_end
          FROM prod.detection.viewing_commercials_firehose AS comms
          JOIN null_gaps_to_fill AS tv
            ON tv.fk_tvid = comms.fk_tvid
           AND comms.session_start <= tv.session_end
           AND comms.session_end >= tv.session_start
          WHERE comms.session_start >= '2025-08-04 00:00:00'
            AND comms.session_start < '2025-08-07 18:00:00'
            AND comms.fk_tvid = 170122240
          GROUP BY ALL
          ORDER BY 1, 2, 3
        ) 
      )
    )
    GROUP BY 1, 2
  )
  SELECT *
  FROM (
    SELECT fk_tvid, session_start, session_end, session_duration, fk_input_source_id
      , media_time_start, media_time_end, is_live
      , runtime, airdate, fk_show_id, fk_station_id, tms_airdate, tms_show_id, tms_station_id, fk_content_id
      , SUM(comm_duration)*1.0 AS total_comm_duration
    FROM (
      SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration, content.fk_input_source_id
      , content.media_time_start, content.media_time_end, content.is_live, content.fk_show_id, content.airdate
      , content.runtime, content.fk_input_source_id
      , content.fk_station_id, content.tms_airdate, content.tms_show_id, content.tms_station_id, content.fk_content_id
      , GREATEST(comms.session_start, content.session_start) AS comm_start
      , LEAST(comms.session_end, content.session_end) AS comm_end
      , TIMESTAMPDIFF(SECOND, comm_start, comm_end) AS comm_duration
      FROM null_gaps_to_fill AS content
      JOIN comms
        ON content.fk_tvid = comms.fk_tvid
      AND comms.session_start <= content.session_end
      AND comms.session_end >= content.session_start
      WHERE content.fk_show_id IS NOT NULL
        AND content.airdate IS NOT NULL
        AND content.media_time_start IS NOT NULL
        AND content.fk_station_id IS NOT NULL
    )
    GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16
  )
  WHERE total_comm_duration > 0
  -- WHERE CASE WHEN session_duration <= 15 THEN total_comm_duration >= 5
  --           ELSE total_comm_duration/session_duration >= 0.5 END
)
SELECT c.fk_tvid
, show.title
, st.inscape_station_name
, REPLACE(NVL(n.airdate, c.airdate), '.000+00:00', '') AS airdate
, REPLACE(c.session_start, '.000+00:00', '') AS session_start
, REPLACE(c.session_end, '.000+00:00', '') AS session_end
, c.session_duration
, NVL(n.media_time_start, c.media_time_start) AS media_time_start
, NVL(n.media_time_end, c.media_time_end) AS media_time_end
, NVL(n.runtime, c.runtime) AS runtime
, NVL(n.fk_content_id, c.fk_content_id) AS fk_content_id
, c.fk_input_source_id
, NVL(n.is_live, c.is_live) AS is_live
, c.vizio_epg_program
, c.tuner_program_id
, CASE WHEN n.fk_tvid IS NOT NULL THEN 1 ELSE 0 END AS is_null_gap
, total_comm_duration
FROM prod.detection.viewing_content_firehose c
LEFT JOIN filled_null_gaps n
  ON n.fk_tvid = c.fk_tvid
 AND n.session_start = c.session_start
 AND n.session_end = c.session_end
 AND n.fk_input_source_id = c.fk_input_source_id
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = NVL(n.fk_station_id, c.fk_station_id)
 AND st.vendor_name = 'TIVO'
LEFT JOIN prod.detection.epg_show show
  ON show.show_id = NVL(n.fk_show_id, c.fk_show_id)
 AND show.vendor_name = 'TIVO'
WHERE c.session_start >= '2025-08-04 00:00:00'
  AND c.session_start < '2025-08-07 00:00:00'
  AND c.fk_tvid = 170122240
ORDER BY c.session_start

In [0]:
WITH null_gaps_to_fill AS (
    SELECT *
    FROM (
      SELECT fk_tvid, session_start, session_end, session_duration, fk_show_id, fk_input_source_id
      , lead(session_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_start
      , lead(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_end
      , LAG(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_end
      , LEAD(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_show_id
      , LAG(fk_show_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_show_id
      , LEAD(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_input_source_id
      , LAG(fk_input_source_id) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_input_source_id
      , LEAD(media_time_start) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_media_time_start
      , LAG(media_time_end) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_media_time_end
      , LEAD(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_runtime
      , LAG(runtime) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_runtime
      , LEAD(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS next_airdate
      , LAG(airdate) OVER (PARTITION BY fk_tvid ORDER BY session_start) AS prev_airdate
      , TIMESTAMPADD(SECOND, prev_runtime + 120, prev_airdate) AS prev_airdate_end
      , TIMESTAMPADD(SECOND, next_runtime + 120, next_airdate) AS next_airdate_end
      , CASE WHEN prev_end <= prev_airdate_end THEN TRUE ELSE FALSE END AS prev_is_live
      , CASE WHEN next_end <= next_airdate_end THEN TRUE ELSE FALSE END AS next_is_live
      FROM prod.detection.viewing_content_firehose
      WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
        AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
    )
    WHERE fk_show_id IS NULL
      AND session_duration <= 180
      AND next_start >= session_end
      AND prev_end <= session_start
      AND CASE WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id = next_show_id
            THEN prev_input_source_id = next_input_source_id AND prev_input_source_id = fk_input_source_id
            AND prev_media_time_end + session_duration <= next_media_time_start
            AND next_is_live
            AND prev_is_live
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
          WHEN prev_show_id IS NOT NULL AND next_show_id IS NULL
            THEN prev_input_source_id <=> fk_input_source_id
            AND prev_media_time_end + session_duration <= prev_runtime
            AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
            AND prev_is_live
          WHEN next_show_id IS NOT NULL AND prev_show_id IS NULL
            THEN next_input_source_id <=> fk_input_source_id
            AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
            AND next_media_time_start - session_duration >= 0
            AND next_is_live
          WHEN prev_show_id IS NOT NULL AND next_show_id IS NOT NULL AND prev_show_id != next_show_id
            THEN CASE WHEN prev_media_time_end + session_duration <= prev_runtime 
                        THEN prev_input_source_id <=> fk_input_source_id
                          AND TIMESTAMPDIFF(SECOND, prev_end, session_start) <= 2
                          AND prev_is_live
                      WHEN next_media_time_start - session_duration >= 0
                        THEN next_input_source_id <=> fk_input_source_id
                        AND TIMESTAMPDIFF(SECOND, session_end, next_start) <= 2
                        AND next_is_live
                      ELSE FALSE END
          ELSE FALSE END
    GROUP BY ALL
  )
, tvids_from_it AS (
  SELECT fk_tvid
  FROM null_gaps_to_fill
  GROUP BY 1
)
, comms AS (
  SELECT fk_tvid, tv_group_num
  , MIN(session_start) AS session_start
  , MAX(session_end) AS session_end
  , TIMESTAMPDIFF(SECOND,  MIN(session_start), MAX(session_end)) AS comm_duration
  FROM (
    SELECT *
    , SUM(tvog_calc) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS UNBOUNDED PRECEDING) AS tv_group_num
    FROM (
      SELECT fk_tvid, session_start, session_end
      , MAX(session_end) OVER (PARTITION BY fk_tvid ORDER BY session_start, session_end ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_tv_prev_end
      , CASE WHEN TIMESTAMPADD(SECOND, -2, session_start) >= max_tv_prev_end OR max_tv_prev_end IS NULL THEN 1 ELSE 0 END AS tvog_calc
      FROM (
        SELECT vc.fk_tvid, vc.session_start, vc.session_end
        FROM prod.detection.viewing_commercials_firehose AS vc
        JOIN tvids_from_it AS tv
          ON tv.fk_tvid = vc.fk_tvid
        WHERE session_start >= DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 4 HOURS
          AND session_start < DATE_TRUNC('HOUR', CURRENT_TIMESTAMP) - INTERVAL 3 HOURS
        GROUP BY ALL
        ORDER BY 1, 2, 3
      ) 
    )
  )
  GROUP BY 1, 2
)
SELECT CASE WHEN session_duration <= 15 THEN 'a - <= 15 Second'
            WHEN session_duration <= 30 THEN 'b - 16-30 Seconds'
            WHEN session_duration <= 45 THEN 'c - 31-45 Seconds'
            WHEN session_duration <= 60 THEN 'd - 46-60 Seconds'
            WHEN session_duration <= 75 THEN 'e - 61-75 Seconds'
            WHEN session_duration <= 90 THEN 'f - 76-90 Seconds'
            WHEN session_duration <= 105 THEN 'g - 91-105 Seconds'
            WHEN session_duration <= 120 THEN 'h - 106-120 Seconds'
            WHEN session_duration <= 135 THEN 'i - 121-135 Seconds'
            WHEN session_duration <= 150 THEN 'j - 136-150 Seconds'
            WHEN session_duration <= 175 THEN 'k - 151-175 Seconds'
            ELSE 'l - 175-180 Seconds' END AS Session_duration_bucket
, CASE WHEN perc_covered < 10 THEN '< 10%'
       WHEN perc_covered < 25 THEN '10%-25%'
       WHEN perc_covered < 40 THEN '25%-40%'
       WHEN perc_covered <= 45 THEN '40%-50%'
       ELSE '50%+' END AS comm_duration_bucket
-- , CASE WHEN total_comm_duration <= 15 THEN '<= 15 Second'
--        WHEN total_comm_duration <= 30 THEN '16-30 Seconds'
--        WHEN total_comm_duration <= 45 THEN '31-45 Seconds'
--        WHEN total_comm_duration <= 60 THEN '46-60 Seconds'
--        WHEN total_comm_duration <= 75 THEN '61-75 Seconds'
--        WHEN total_comm_duration <= 90 THEN '76-90 Seconds'
--        WHEN total_comm_duration <= 105 THEN '91-105 Seconds'
--        WHEN total_comm_duration <= 120 THEN '106-120 Seconds'
--        WHEN total_comm_duration <= 135 THEN '121-135 Seconds'
--        WHEN total_comm_duration <= 150 THEN '136-150 Seconds'
--        WHEN total_comm_duration <= 175 THEN '151-175 Seconds'
--        ELSE '175-180 Seconds' END AS comm_duration_bucket
, COUNT(*) AS session_count
FROM (
  SELECT fk_tvid, session_start, session_end, session_duration, SUM(comm_duration)*1.0 AS total_comm_duration,  (SUM(comm_duration)*1.0/session_duration)*100 AS perc_covered
    FROM (
      SELECT content.fk_tvid, content.session_start, content.session_end, content.session_duration
      , GREATEST(comms.session_start, content.session_start) AS comm_start
      , LEAST(comms.session_end, content.session_end) AS comm_end
      , TIMESTAMPDIFF(SECOND, comm_start, comm_end)*1.0 AS comm_duration
      FROM null_gaps_to_fill AS content
      JOIN comms
        ON content.fk_tvid = comms.fk_tvid
      AND comms.session_start <= content.session_end
      AND comms.session_end >= content.session_start
    )
  GROUP BY 1, 2, 3, 4)
  WHERE total_comm_duration > 0
GROUP BY 1, 2

In [0]:
SELECT fk_tvid
, show.title
, st.inscape_station_name
, REPLACE(c.airdate, '.000+00:00', '') AS airdate
, REPLACE(session_start, '.000+00:00', '') AS session_start
, REPLACE(session_end, '.000+00:00', '') AS session_end
, session_duration, media_time_start, media_time_end, runtime
, fk_content_id, fk_input_source_id, is_live, vizio_epg_program, tuner_program_id
FROM prod.detection.viewing_content_firehose c
LEFT JOIN prod.detection.epg_station st
  ON st.station_id = c.fk_station_id
 AND st.vendor_name = 'TIVO'
LEFT JOIN prod.detection.epg_show show
  ON show.show_id = c.fk_show_id
 AND show.vendor_name = 'TIVO'
WHERE fk_tvid = 170122240
AND DATE_TRUNC('HOUR', session_start) >= '2025-08-04 00:00:00'
ORDER BY session_start

In [0]:
SELECT process_hr, tv_count_content, content_count
FROM prod.cooker.hourly_metrics
WHERE process_hr >= CURRENT_DATE

In [0]:
SELECT DATE_TRUNC('HOUR', session_start), COUNT(DISTINCT fk_tvid), COUNT(*)
FROM prod.detection.viewing_content_firehose
WHERE session_start >= CURRENT_DATE
GROUP BY 1

In [0]:
WITH inscape_station_map_dedupe AS (
  SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id, channel_affiliate
  FROM (
    SELECT inscape_station_id, inscape_call_sign, mapped_vendor, mapped_vendor_station_id
    , CASE WHEN st.inscape_station_name IS NOT NULL THEN st.inscape_station_name
           WHEN LOWER(st.station_affil) LIKE '%affiliate%'
             OR LOWER(st.station_affil) LIKE '%independent%'
             OR LOWER(st.station_affil) LIKE '%low power%' THEN st.station_affil
      END AS channel_affiliate
    , ROW_NUMBER() OVER (PARTITION BY ism.mapped_vendor, ism.mapped_vendor_station_id ORDER BY ism.created_at DESC) AS rn
    FROM prod.detection.inscape_station_map ism
    JOIN prod.detection.epg_station st
      ON st.station_id = ism.mapped_vendor_station_id
     AND st.vendor_name = ism.mapped_vendor
  ) ism
  WHERE ism.rn = 1
)
SELECT DATE_TRUNC('HOUR', c.session_start), COUNT(DISTINCT c.fk_tvid), COUNT(*)
FROM prod.detection.viewing_content_firehose AS c
  JOIN detection.zoo AS z
    ON c.fk_zoo_id = z.zoo_id
   AND z.zoo = 'control-zoo-dtsprod.tvinteractive.tv'
  JOIN detection.tv AS tv
    ON c.fk_tvid = tv.tvid
   AND tv.oem = 'VIZIO'
  -- Location
  JOIN detection.tv_settings AS tv_settings
    ON c.session_start >= tv_settings.create_timestamp
   AND c.session_start < tv_settings.next_create_timestamp
   AND tv_settings.create_timestamp <= CURRENT_DATE + 1
   AND tv_settings.next_create_timestamp >= CURRENT_DATE
   AND c.fk_tvid = tv_settings.fk_tvid
  JOIN detection.settings AS settings
    ON tv_settings.fk_settings_id = settings.settings_id
   AND UPPER(settings.country_name) = 'USA'
  JOIN detection.tv_populations AS u
    ON c.fk_tvid = u.fk_tvid
  JOIN detection.populations AS pop
    ON u.fk_population_id = pop.population_id
   AND pop.population_name = 'opted_in'
  JOIN detection.location AS location
    ON c.fk_location_id = location.location_id
   AND UPPER(location.country_code) = 'US'
  JOIN detection.content_ids_firehose AS cid
    ON cid.content_id = c.fk_content_id
  JOIN detection.tv_input_stats_firehose AS tvis
    ON c.session_start >= tvis.create_timestamp
   AND c.session_start < tvis.next_create_timestamp
   AND tvis.create_timestamp <= CURRENT_DATE + 1
   AND tvis.next_create_timestamp >= CURRENT_DATE
   AND c.fk_tvid = tvis.fk_tvid
   AND c.fk_input_source_id = tvis.fk_input_source_id
  LEFT OUTER JOIN inscape_station_map_dedupe AS tivo_map
    ON tivo_map.mapped_vendor_station_id = c.fk_station_id
   AND tivo_map.mapped_vendor = 'TIVO'
  LEFT OUTER JOIN inscape_station_map_dedupe AS tms_map
    ON tms_map.mapped_vendor_station_id = c.tms_station_id
   AND tms_map.mapped_vendor = 'TMS'
  ---------------------------------------------
WHERE c.session_start >= CURRENT_DATE
  AND c.session_start < CURRENT_DATE + 1
    AND CASE c.file_ingested
      WHEN true THEN
          CASE NULLIF(SPLIT(cid.content_cid, '_')[1], '') IS NOT NULL AND NULLIF(SPLIT_PART(cid.content_cid, '_', 3), '') IS NULL
          WHEN true THEN SPLIT(cid.content_cid, '_')[1]
          ELSE NULL
          END
      ELSE COALESCE(tivo_map.inscape_call_sign, tms_map.inscape_call_sign, 'KeepSessionForNullReport')
      END NOT IN (SELECT DISTINCT chan_callsign FROM customer_reports.bad_chan_callsign)
GROUP BY 1

In [0]:
SELECT DATE_TRUNC('HOUR', c.session_start), COUNT(DISTINCT c.fk_tvid), COUNT(*)
FROM prod.detection.viewing_content_golden c
WHERE c.session_start >= CURRENT_DATE
  AND c.session_start < CURRENT_DATE + 1
GROUP BY 1

In [0]:
SELECT created_at, DATE_TRUNC('HOUR', session_start), COUNT(DISTINCT fk_tvid), COUNT(*)
FROM prod.detection.viewing_content_firehose c
WHERE c.session_start >= CURRENT_DATE
  AND c.session_start < CURRENT_DATE + 1
GROUP BY 1, 2

In [0]:
SELECT * FROM detection.tv_populations
WHERE create_timestamp >= CURRENT_DATE
-- AND next_create_timestamp IS NULL
ORDER BY create_timestamp DESC
LIMIT 100